# Personalized Federated Learning for Transport Mode Detection (CellMob)

This notebook contains **both**:

- **Part 1 — Legacy (midterm report):** **SimpleNet** (two ×128 hidden), batch `64`, `META_EPOCHS=20`, FedAvg / local-only / **Per-FedAvg** / **FedPer**, ablations, per-class metrics (sections from **Data preparation** through the results tables).
- **Part 2 — Unified team protocol:** **TMDNet** (`hidden_dim=64`), batch `256`, `num_rounds=5`, etc. — same logic as `train_unified_*_tmdnet.py` — **run the cells at the end** of this notebook; results save to `results_unified/`.

Supporting modules live in this folder: `unified_team_protocol.py`, `unified_cellmob_data.py`, `unified_results_log.py`, `README_UNIFIED.md`.


## Where everything lives in this notebook

| Block | Content |
|-------|--------|
| **Above** (from Data preparation onward) | Legacy **SimpleNet** experiments and all report figures/tables. |
| **Bottom** (section *Unified experiments — TMDNet*) | Imports `train_unified_fedavg_tmdnet` / `train_unified_per_fedavg_tmdnet` and runs the same training **inside the notebook**; JSON also written to `results_unified/`. |

**Team agreement (unified):** `TMDNet(input_dim=len(feature_cols), hidden_dim=64, num_classes=...)`; batch `256`; `max_per_client=100000`; `local_epochs=1`; `num_rounds=5`; shared pipeline → `X_test_fl`, `y_test_fl` via `get_federated_test_tensors`.


In [19]:
from pathlib import Path
from typing import Dict, List, Tuple, Any

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, accuracy_score

PROJECT_ROOT = Path.cwd()
CLIENTS_ROOT = PROJECT_ROOT / "cellmob_clients"
CLIENT_FILES = [CLIENTS_ROOT / f"client_{i}.csv" for i in range(1, 5)]

print("Project root:", PROJECT_ROOT)
print("Client files:")
for p in CLIENT_FILES:
    print("  ", p)


Project root: /Users/chelseasaydi/Personalized Federated Learning/Personalized_Federated_Learning_Implementation
Client files:
   /Users/chelseasaydi/Personalized Federated Learning/Personalized_Federated_Learning_Implementation/cellmob_clients/client_1.csv
   /Users/chelseasaydi/Personalized Federated Learning/Personalized_Federated_Learning_Implementation/cellmob_clients/client_2.csv
   /Users/chelseasaydi/Personalized Federated Learning/Personalized_Federated_Learning_Implementation/cellmob_clients/client_3.csv
   /Users/chelseasaydi/Personalized Federated Learning/Personalized_Federated_Learning_Implementation/cellmob_clients/client_4.csv


In [20]:
# Add Implementation/ to path for Part 2 (unified TMDNet) cells at notebook end.
import sys
from pathlib import Path
_cwd = Path.cwd().resolve()
_impl = _cwd / "Personalized_Federated_Learning_Implementation"
if (_impl / "unified_team_protocol.py").is_file():
    IMPLEMENTATION_DIR = _impl
elif (_cwd / "unified_team_protocol.py").is_file():
    IMPLEMENTATION_DIR = _cwd
else:
    raise FileNotFoundError("Run from repo root or Personalized_Federated_Learning_Implementation/")
if str(IMPLEMENTATION_DIR) not in sys.path:
    sys.path.insert(0, str(IMPLEMENTATION_DIR))
print("IMPLEMENTATION_DIR =", IMPLEMENTATION_DIR)


IMPLEMENTATION_DIR = /Users/chelseasaydi/Personalized Federated Learning/Personalized_Federated_Learning_Implementation


## Data preparation

We convert each client CSV into window-level samples:

- Group rows by `window_id`.
- For each numeric feature, take the **mean** over the window.
- Assign the window label as the **majority `mode`** in that window.
- Perform an 80/20 train/test split over windows for each client.


In [21]:
TRAIN_RATIO = 0.8

class WindowDataset(Dataset):
    def __init__(self, features: np.ndarray, labels: np.ndarray):
        self.x = torch.from_numpy(features).float()
        self.y = torch.from_numpy(labels).long()

    def __len__(self) -> int:
        return self.x.shape[0]

    def __getitem__(self, idx: int):
        return self.x[idx], self.y[idx]


# Build a **global** mapping from transport mode string -> integer class index
# so that the classifier's output dimension has a consistent semantic meaning
# across all cities/clients.
GLOBAL_MODE_TO_IDX: Dict[str, int] = {}
for csv_path in CLIENT_FILES:
    df_modes = pd.read_csv(csv_path, usecols=["mode"], low_memory=False)
    for m in df_modes["mode"].dropna().astype(str).unique():
        if m not in GLOBAL_MODE_TO_IDX:
            GLOBAL_MODE_TO_IDX[m] = len(GLOBAL_MODE_TO_IDX)

print("Global mode-to-index mapping (shared across clients):")
for mode, idx in GLOBAL_MODE_TO_IDX.items():
    print(f"  {mode!r} -> {idx}")


def load_client_windows(
    csv_path: Path,
    feature_cols: List[str] | None = None,
    mode_to_idx: Dict[str, int] | None = None,
) -> Tuple[np.ndarray, np.ndarray, List[str]]:
    """Load one client CSV and build (X, y) at window level.

    - X: mean features per window
    - y: integer-encoded transport mode per window, using a *global* mapping
    """
    if mode_to_idx is None:
        mode_to_idx = GLOBAL_MODE_TO_IDX

    df = pd.read_csv(csv_path, low_memory=False)

    if "window_id" not in df.columns or "mode" not in df.columns:
        raise ValueError(f"CSV {csv_path} is missing 'window_id' or 'mode' columns.")

    non_feature_cols = {"Time", "city", "mode"}
    numeric_df = df.copy()
    for col in numeric_df.columns:
        if col in non_feature_cols:
            continue
        numeric_df[col] = pd.to_numeric(numeric_df[col], errors="coerce")

    if feature_cols is None:
        inferred_feature_cols: List[str] = [
            c
            for c in numeric_df.columns
            if c not in non_feature_cols and c != "window_id" and pd.api.types.is_numeric_dtype(numeric_df[c])
        ]
        if not inferred_feature_cols:
            raise ValueError(f"No numeric feature columns found in {csv_path}")
        feature_cols = inferred_feature_cols

    grouped = numeric_df.groupby("window_id")
    window_features: List[np.ndarray] = []
    window_labels: List[int] = []

    for window_id, g in grouped:
        modes = g["mode"].dropna().astype(str)
        if modes.empty:
            continue

        mode = modes.mode().iloc[0]
        if mode not in mode_to_idx:
            # Skip windows with unseen/unknown mode labels
            continue

        feats = g[feature_cols].mean(axis=0).to_numpy(dtype=np.float32)
        if np.any(np.isnan(feats)):
            continue

        window_features.append(feats)
        window_labels.append(mode_to_idx[mode])

    if not window_features:
        raise ValueError(f"No valid windows found in {csv_path}")

    X = np.stack(window_features, axis=0)
    y = np.array(window_labels, dtype=np.int64)
    return X, y, feature_cols


# First pass: infer common numeric feature columns across the 4 main clients
per_client_features: Dict[str, List[str]] = {}
for csv_path in CLIENT_FILES:
    tmp_df = pd.read_csv(csv_path, low_memory=False)
    non_feature_cols = {"Time", "city", "mode"}
    for col in tmp_df.columns:
        if col in non_feature_cols or col == "window_id":
            continue
        tmp_df[col] = pd.to_numeric(tmp_df[col], errors="coerce")
    feature_cols = [
        c
        for c in tmp_df.columns
        if c not in non_feature_cols and c != "window_id" and pd.api.types.is_numeric_dtype(tmp_df[c])
    ]
    if not feature_cols:
        raise ValueError(f"No numeric feature columns found in {csv_path}")
    per_client_features[csv_path.stem] = feature_cols

common_features = set(next(iter(per_client_features.values())))
for feats in per_client_features.values():
    common_features &= set(feats)
common_features = sorted(common_features)

print(f"Number of common numeric feature columns across 4 clients: {len(common_features)}")

# Second pass: actually build datasets using these common features
clients: Dict[str, Dict[str, WindowDataset]] = {}
input_dim = len(common_features)

global_num_classes = len(GLOBAL_MODE_TO_IDX)
print("Global number of classes (modes):", global_num_classes)

for csv_path in CLIENT_FILES:
    client_name = csv_path.stem
    print(f"Loading {client_name} from {csv_path} ...")
    X, y, _ = load_client_windows(csv_path, feature_cols=common_features, mode_to_idx=GLOBAL_MODE_TO_IDX)

    n = X.shape[0]
    split = int(TRAIN_RATIO * n)
    if split <= 0 or split >= n:
        split = max(1, n - 1)

    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    num_classes_client = int(y.max()) + 1

    clients[client_name] = {
        "train": WindowDataset(X_train, y_train),
        "test": WindowDataset(X_test, y_test),
    }

    print(
        f"  {client_name}: {X_train.shape[0]} train windows, "
        f"{X_test.shape[0]} test windows, {num_classes_client} classes"
    )

client_names = sorted(clients.keys())
print("Input dimension:", input_dim)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/chelseasaydi/Personalized Federated Learning/Personalized_Federated_Learning_Implementation/cellmob_clients/client_1.csv'

## CSV data summary (size and heterogeneity)

One table summarizing each client’s data: train/test window counts and label distribution (%). Shows data size and label heterogeneity across clients.

In [ ]:
# Build summary table: all clients in CLIENT_FILES; rows = estimated from windows (windows × 256, match prepare_cellmob_clients)
ROWS_PER_WINDOW = 256
IDX_TO_MODE = {v: k for k, v in GLOBAL_MODE_TO_IDX.items()}
summary_rows = []
for csv_path in sorted(CLIENT_FILES):
    cname = csv_path.stem
    row = {"Client": cname}
    if cname in clients:
        train_ds = clients[cname]["train"]
        test_ds = clients[cname]["test"]
        n_train, n_test = len(train_ds), len(test_ds)
        total_windows = n_train + n_test
        row["CSV rows"] = total_windows * ROWS_PER_WINDOW
        all_y = torch.cat([train_ds.y, test_ds.y]).numpy()
        n = len(all_y)
        dist = {}
        for idx in range(global_num_classes):
            name = IDX_TO_MODE.get(idx, str(idx))
            dist[name] = (all_y == idx).sum() / n if n > 0 else 0.0
        n_classes = sum(1 for p in dist.values() if p > 0)
        row["Train"] = n_train
        row["Test"] = n_test
        row["Total"] = total_windows
        row["Classes"] = n_classes
        row.update({f"{m} (%)": f"{100*dist[m]:.0f}" for m in dist})
    else:
        # Client not in memory (e.g. main load failed for this one): load it here for the table
        try:
            X, y, _ = load_client_windows(csv_path, feature_cols=common_features, mode_to_idx=GLOBAL_MODE_TO_IDX)
            n = X.shape[0]
            split = int(TRAIN_RATIO * n)
            if split <= 0 or split >= n:
                split = max(1, n - 1)
            n_train, n_test = split, n - split
            row["CSV rows"] = n * ROWS_PER_WINDOW
            dist = {}
            for idx in range(global_num_classes):
                name = IDX_TO_MODE.get(idx, str(idx))
                dist[name] = (y == idx).sum() / n if n > 0 else 0.0
            n_classes = sum(1 for p in dist.values() if p > 0)
            row["Train"] = n_train
            row["Test"] = n_test
            row["Total"] = n
            row["Classes"] = n_classes
            row.update({f"{m} (%)": f"{100*dist[m]:.0f}" for m in dist})
        except Exception as e:
            row["CSV rows"] = row["Train"] = row["Test"] = row["Total"] = row["Classes"] = "-"
            for m in IDX_TO_MODE.values():
                row[f"{m} (%)"] = "-"
    summary_rows.append(row)
csv_summary = pd.DataFrame(summary_rows).set_index("Client")
print("Data summary: Rows (est. = windows×256), window counts, and label distribution (%) per client (heterogeneity)\n")
display(csv_summary)

## Heterogeneity quantification

We quantify client drift (non-IID) to motivate personalized FL:
- **Label distribution** per client (share of each transport mode)
- **Data size** (train/test windows) and **number of classes** per client

In [ ]:
# Build reverse mapping: index -> mode name for reporting
IDX_TO_MODE = {v: k for k, v in GLOBAL_MODE_TO_IDX.items()}

def get_client_label_distribution(client_name: str) -> Dict[str, float]:
    """Return fraction of each class (by name) in client's combined train+test."""
    train_ds = clients[client_name]["train"]
    test_ds = clients[client_name]["test"]
    all_y = torch.cat([train_ds.y, test_ds.y]).numpy()
    n = len(all_y)
    dist = {}
    for idx in range(global_num_classes):
        name = IDX_TO_MODE.get(idx, str(idx))
        dist[name] = (all_y == idx).sum() / n if n > 0 else 0.0
    return dist

# Per-client label distribution and sizes (for sample-weighted accuracy)
client_label_dists = {}
client_test_sizes = {}
client_train_sizes = {}
print("Label distribution per client (fraction per transport mode):")
print("-" * 60)
for cname in client_names:
    client_label_dists[cname] = get_client_label_distribution(cname)
    n_test = len(clients[cname]["test"])
    n_train = len(clients[cname]["train"])
    client_test_sizes[cname] = n_test
    client_train_sizes[cname] = n_train
    dist_str = ", ".join(f"{mode}: {p:.2f}" for mode, p in client_label_dists[cname].items())
    print(f"  {cname}: train={n_train}, test={n_test} | {dist_str}")
total_test = sum(client_test_sizes.values())
print("-" * 60)
print(f"Total test samples: {total_test}")

Label distribution per client (fraction per transport mode):
------------------------------------------------------------
  client_1: train=10, test=3 | car: 0.23, walk: 0.46, bus: 0.31, train: 0.00
  client_2: train=38, test=10 | car: 0.96, walk: 0.04, bus: 0.00, train: 0.00
  client_3: train=751, test=188 | car: 0.48, walk: 0.18, bus: 0.35, train: 0.00
  client_4: train=806, test=202 | car: 0.16, walk: 0.30, bus: 0.48, train: 0.06
------------------------------------------------------------
Total test samples: 403


## Data summary table (client heterogeneity)

Compact view of each client's data size and label distribution — highlights non-IID (different sizes and mode mixes across clients).

In [ ]:
# Build summary table: one row per client, columns = sizes + label fractions
summary_rows = []
for cname in client_names:
    n_train = client_train_sizes[cname]
    n_test = client_test_sizes[cname]
    dist = client_label_dists[cname]
    n_classes = sum(1 for p in dist.values() if p > 0)
    row = {
        "Client": cname,
        "Train": n_train,
        "Test": n_test,
        "Total": n_train + n_test,
        "Classes": n_classes,
        **{f"{m} (%)": f"{100*dist[m]:.0f}" for m in dist},
    }
    summary_rows.append(row)

data_summary = pd.DataFrame(summary_rows).set_index("Client")
print("Data summary: CSV-derived windows per client and label distribution (heterogeneity)\n")
display(data_summary)

Data summary: CSV-derived windows per client and label distribution (heterogeneity)



,Train,Test,Total,Classes,car (%),walk (%),bus (%),train (%)
Client,,,,,,,,
client_1,10,3,13,3,23,46,31,0
client_2,38,10,48,2,96,4,0,0
client_3,751,188,939,3,48,18,35,0
client_4,806,202,1008,4,16,30,48,6


## Model and Per-FedAvg-style training

We now define a simple fully-connected neural network and a first-order Per-FedAvg-like training loop:

- **Inner loop**: for each client, adapt the model on its training windows with a few SGD steps.
- **Outer loop (meta-update)**: compute gradients of the client loss at the adapted model and average across clients.
- The averaged gradient updates the shared initialization.

At evaluation time, we personalize again (inner steps on train) and then measure test accuracy per client.

**Group comparison:** Run **`train_unified_fedavg_tmdnet.py`** (and future `train_unified_*` scripts) with **`unified_team_protocol.py`** / **`unified_cellmob_data.py`** — see **`README_UNIFIED.md`**. The training code in cells below stays **legacy** (**`SimpleNet`**, batch 64, 20 rounds) for the reported midterm results.


In [ ]:
import copy
import random

# --- Hyperparameters (used by all methods) ---
META_EPOCHS = 20
INNER_STEPS = 3
INNER_LR = 1e-2
META_LR = 1e-3
FEDAVG_LR = 1e-3
LOCAL_EPOCHS = 30
BATCH_SIZE = 64
SEEDS = [42, 123, 456]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
client_names = sorted(clients.keys())

# --- Model architectures ---
class SimpleNet(nn.Module):
    def __init__(self, input_dim: int, num_classes: int, hidden: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class FedPerBody(nn.Module):
    """Shared body: input -> hidden -> hidden -> embedding (no final layer)."""
    def __init__(self, input_dim: int, hidden: int = 128, embed_dim: int = 64):
        super().__init__()
        self.embed_dim = embed_dim
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, embed_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def get_fedper_full_model(body: FedPerBody, head: nn.Linear, device: torch.device) -> nn.Module:
    """Return a single module that runs body then head (for evaluation)."""
    class Wrapper(nn.Module):
        def __init__(self, b, h):
            super().__init__()
            self.body, self.head = b, h
        def forward(self, x):
            return self.head(self.body(x))
    return Wrapper(body, head).to(device)


# --- Evaluation: global model (no personalization) ---
def evaluate_global(
    model: nn.Module,
    test_ds: Dataset,
    device: torch.device,
) -> float:
    model.eval()
    loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)
    correct, total = 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            preds = logits.argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += yb.numel()
    return correct / total if total > 0 else 0.0


# --- Evaluation: personalized (inner-loop adaptation then test) ---
def evaluate_personalized(
    model: nn.Module,
    train_ds: Dataset,
    test_ds: Dataset,
    inner_steps: int,
    inner_lr: float,
    device: torch.device,
) -> float:
    local = copy.deepcopy(model).to(device)
    opt = torch.optim.SGD(local.parameters(), lr=inner_lr)
    loss_fn = nn.CrossEntropyLoss()
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    for _ in range(inner_steps):
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = loss_fn(local(xb), yb)
            loss.backward()
            opt.step()
    return evaluate_global(local, test_ds, device)


# --- Detailed metrics (per-class P/R/F1, confusion matrix, weighted acc) ---
def get_detailed_metrics(
    model: nn.Module,
    test_ds: Dataset,
    device: torch.device,
    train_ds: Dataset | None = None,
    inner_steps: int = 0,
    inner_lr: float = 1e-2,
    num_classes: int | None = None,
) -> Dict[str, Any]:
    if train_ds is not None and inner_steps > 0:
        model = copy.deepcopy(model).to(device)
        opt = torch.optim.SGD(model.parameters(), lr=inner_lr)
        loss_fn = nn.CrossEntropyLoss()
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        for _ in range(inner_steps):
            for xb, yb in train_loader:
                xb, yb = xb.to(device), yb.to(device)
                opt.zero_grad()
                loss = loss_fn(model(xb), yb)
                loss.backward()
                opt.step()
    model.eval()
    loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)
    all_preds, all_y = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            all_preds.append(model(xb).argmax(dim=1).cpu().numpy())
            all_y.append(yb.numpy())
    y_true = np.concatenate(all_y)
    y_pred = np.concatenate(all_preds)
    acc = accuracy_score(y_true, y_pred)
    labels = np.arange(num_classes) if num_classes is not None else None
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, average=None, zero_division=0
    )
    cm = confusion_matrix(y_true, y_pred, labels=labels) if labels is not None else confusion_matrix(y_true, y_pred)
    return {
        "accuracy": acc,
        "y_true": y_true,
        "y_pred": y_pred,
        "precision_per_class": p,
        "recall_per_class": r,
        "f1_per_class": f1,
        "confusion_matrix": cm,
    }


def sample_weighted_accuracy(per_client_acc: Dict[str, float], test_sizes: Dict[str, int]) -> float:
    total = sum(test_sizes.values())
    if total == 0:
        return 0.0
    return sum(per_client_acc[k] * test_sizes[k] for k in per_client_acc) / total


Using device: cpu


## Training functions: FedAvg, Local-only, Per-FedAvg, FedPer

Each function takes a `seed` and returns per-client test accuracies (and optionally the model(s)) so we can aggregate over seeds.

In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def train_vanilla_fedavg(seed: int) -> Dict[str, float]:
    """Vanilla FedAvg: each round, each client does local SGD from global model; server averages params."""
    set_seed(seed)
    global_model = SimpleNet(input_dim, global_num_classes).to(device)
    loss_fn = nn.CrossEntropyLoss()
    for epoch in range(1, META_EPOCHS + 1):
        client_states = []
        for cname in client_names:
            local = copy.deepcopy(global_model).to(device)
            opt = torch.optim.SGD(local.parameters(), lr=FEDAVG_LR)
            loader = DataLoader(clients[cname]["train"], batch_size=BATCH_SIZE, shuffle=True)
            local.train()
            for _ in range(INNER_STEPS):
                for xb, yb in loader:
                    xb, yb = xb.to(device), yb.to(device)
                    opt.zero_grad()
                    loss = loss_fn(local(xb), yb)
                    loss.backward()
                    opt.step()
            client_states.append(local.state_dict())
        # Average parameters
        avg_state = {}
        for key in client_states[0]:
            avg_state[key] = torch.stack([client_states[i][key].float() for i in range(len(client_names))]).mean(dim=0)
        global_model.load_state_dict(avg_state)
    accs = {cname: evaluate_global(global_model, clients[cname]["test"], device) for cname in client_names}
    return accs


def train_local_only(seed: int) -> Dict[str, float]:
    """Each client trains only on its own data (no federation). One model per client."""
    set_seed(seed)
    accs = {}
    for cname in client_names:
        model = SimpleNet(input_dim, global_num_classes).to(device)
        opt = torch.optim.Adam(model.parameters(), lr=FEDAVG_LR)
        loss_fn = nn.CrossEntropyLoss()
        loader = DataLoader(clients[cname]["train"], batch_size=BATCH_SIZE, shuffle=True)
        for _ in range(LOCAL_EPOCHS):
            for xb, yb in loader:
                xb, yb = xb.to(device), yb.to(device)
                opt.zero_grad()
                loss = loss_fn(model(xb), yb)
                loss.backward()
                opt.step()
        accs[cname] = evaluate_global(model, clients[cname]["test"], device)
    return accs


def train_per_fedavg(seed: int, inner_steps: int = INNER_STEPS) -> Dict[str, float]:
    """Per-FedAvg: meta-learn shared init, then personalize (inner steps) per client at eval."""
    set_seed(seed)
    model = SimpleNet(input_dim, global_num_classes).to(device)
    meta_opt = torch.optim.Adam(model.parameters(), lr=META_LR)
    loss_fn = nn.CrossEntropyLoss()
    for epoch in range(1, META_EPOCHS + 1):
        model.train()
        meta_opt.zero_grad()
        for cname in client_names:
            train_ds = clients[cname]["train"]
            train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
            local = copy.deepcopy(model).to(device)
            inner_opt = torch.optim.SGD(local.parameters(), lr=INNER_LR)
            for _ in range(inner_steps):
                for xb, yb in train_loader:
                    xb, yb = xb.to(device), yb.to(device)
                    inner_opt.zero_grad()
                    loss = loss_fn(local(xb), yb)
                    loss.backward()
                    inner_opt.step()
            local.train()
            inner_grads = [torch.zeros_like(p) for p in model.parameters()]
            for xb, yb in train_loader:
                xb, yb = xb.to(device), yb.to(device)
                local.zero_grad()
                loss = loss_fn(local(xb), yb)
                loss.backward()
                for k, p in enumerate(local.parameters()):
                    if p.grad is not None:
                        inner_grads[k] = inner_grads[k] + p.grad.detach()
            n_b = len(train_loader)
            if n_b > 0:
                inner_grads = [g / n_b for g in inner_grads]
            for gp, lg in zip(model.parameters(), inner_grads):
                if gp.grad is None:
                    gp.grad = lg.clone()
                else:
                    gp.grad += lg
        for p in model.parameters():
            if p.grad is not None:
                p.grad /= len(client_names)
        meta_opt.step()
    accs = {
        cname: evaluate_personalized(
            model, clients[cname]["train"], clients[cname]["test"],
            inner_steps=inner_steps, inner_lr=INNER_LR, device=device,
        )
        for cname in client_names
    }
    return accs


def train_fedper(seed: int) -> Dict[str, float]:
    """FedPer: shared body + personalized head per client. Body is averaged; heads stay local."""
    set_seed(seed)
    hidden, embed_dim = 128, 64
    body = FedPerBody(input_dim, hidden=hidden, embed_dim=embed_dim).to(device)
    heads = {cname: nn.Linear(embed_dim, global_num_classes).to(device) for cname in client_names}
    body_opt = torch.optim.Adam(body.parameters(), lr=FEDAVG_LR)
    head_opts = {cname: torch.optim.Adam(heads[cname].parameters(), lr=FEDAVG_LR) for cname in client_names}
    loss_fn = nn.CrossEntropyLoss()
    for epoch in range(1, META_EPOCHS + 1):
        body_states = []
        for cname in client_names:
            body.train()
            heads[cname].train()
            loader = DataLoader(clients[cname]["train"], batch_size=BATCH_SIZE, shuffle=True)
            for xb, yb in loader:
                xb, yb = xb.to(device), yb.to(device)
                body_opt.zero_grad()
                head_opts[cname].zero_grad()
                logits = heads[cname](body(xb))
                loss = loss_fn(logits, yb)
                loss.backward()
                body_opt.step()
                head_opts[cname].step()
            body_states.append(copy.deepcopy(body.state_dict()))
        # Average body parameters
        avg_body = {}
        for key in body_states[0]:
            avg_body[key] = torch.stack([body_states[i][key].float() for i in range(len(client_names))]).mean(dim=0)
        body.load_state_dict(avg_body)
    accs = {}
    for cname in client_names:
        full = get_fedper_full_model(body, heads[cname], device)
        accs[cname] = evaluate_global(full, clients[cname]["test"], device)
    return accs

## Multi-seed experiments

Run all four methods with multiple random seeds and collect per-client and average accuracies for reporting (mean ± std).

In [ ]:
# Run all methods for each seed and store results
METHODS = {
    "Vanilla FedAvg": train_vanilla_fedavg,
    "Local-only": train_local_only,
    "Per-FedAvg": train_per_fedavg,
    "FedPer": train_fedper,
}
# results[method_name][seed] = { client_1: acc, ... }
results: Dict[str, Dict[int, Dict[str, float]]] = {name: {} for name in METHODS}

for method_name, train_fn in METHODS.items():
    print(f"Running {method_name} ...")
    for seed in SEEDS:
        accs = train_fn(seed)
        results[method_name][seed] = accs
    print(f"  Done. Example (seed={SEEDS[0]}): avg = {100*np.mean(list(results[method_name][SEEDS[0]].values())):.1f}%")

Running Vanilla FedAvg ...
  Done. Example (seed=42): avg = 16.4%
Running Local-only ...
  Done. Example (seed=42): avg = 39.2%
Running Per-FedAvg ...
  Done. Example (seed=42): avg = 50.0%
Running FedPer ...
  Done. Example (seed=42): avg = 33.6%


## Final comparison report

Per-client and average test accuracy (mean ± std over seeds). **Avg (equal)** = mean over clients; **Avg (weighted)** = sample-weighted by test set size.

In [ ]:
# Aggregate: for each method, compute mean and std per client and for averages
def agg_accs(acc_list: List[Dict[str, float]]) -> Tuple[Dict[str, float], Dict[str, float]]:
    """acc_list = [result_seed1, result_seed2, ...]. Return mean and std per client."""
    keys = acc_list[0].keys()
    mean_acc = {k: np.mean([r[k] for r in acc_list]) for k in keys}
    std_acc = {k: np.std([r[k] for r in acc_list]) for k in keys}
    return mean_acc, std_acc

print("=" * 80)
print("FINAL RESULTS (mean ± std over seeds)")
print("=" * 80)
for method_name in METHODS:
    acc_list = [results[method_name][s] for s in SEEDS]
    mean_a, std_a = agg_accs(acc_list)
    avg_equal = np.mean(list(mean_a.values()))
    avg_equal_std = np.std([np.mean(list(r.values())) for r in acc_list])
    avg_weighted = sample_weighted_accuracy(mean_a, client_test_sizes)
    avg_weighted_std = np.std([sample_weighted_accuracy(r, client_test_sizes) for r in acc_list])
    print(f"\n{method_name}")
    print("-" * 40)
    for cname in client_names:
        print(f"  {cname}: {100*mean_a[cname]:.1f}% ± {100*std_a[cname]:.1f}%")
    print(f"  Avg (equal):    {100*avg_equal:.1f}% ± {100*avg_equal_std:.1f}%")
    print(f"  Avg (weighted): {100*avg_weighted:.1f}% ± {100*avg_weighted_std:.1f}%")
print("=" * 80)

FINAL RESULTS (mean ± std over seeds)

Vanilla FedAvg
----------------------------------------
  client_1: 0.0% ± 0.0%
  client_2: 0.0% ± 0.0%
  client_3: 8.5% ± 0.0%
  client_4: 56.9% ± 0.0%
  Avg (equal):    16.4% ± 0.0%
  Avg (weighted): 32.5% ± 0.0%

Local-only
----------------------------------------
  client_1: 0.0% ± 0.0%
  client_2: 100.0% ± 0.0%
  client_3: 49.5% ± 10.5%
  client_4: 14.4% ± 20.3%
  Avg (equal):    41.0% ± 6.8%
  Avg (weighted): 32.8% ± 13.3%

Per-FedAvg
----------------------------------------
  client_1: 0.0% ± 0.0%
  client_2: 100.0% ± 0.0%
  client_3: 24.6% ± 22.8%
  client_4: 52.3% ± 6.5%
  Avg (equal):    44.2% ± 4.1%
  Avg (weighted): 40.2% ± 7.4%

FedPer
----------------------------------------
  client_1: 0.0% ± 0.0%
  client_2: 66.7% ± 47.1%
  client_3: 33.3% ± 19.8%
  client_4: 38.0% ± 26.8%
  Avg (equal):    34.5% ± 15.2%
  Avg (weighted): 36.2% ± 16.1%


## Exact accuracies for each model

One table with test accuracy (mean ± std over seeds) for each method. Run the **Multi-seed experiments** cell first, then this cell.

In [ ]:
# Build one clear table: rows = models, columns = clients + averages
def _mean_std(acc_list):
    keys = acc_list[0].keys()
    mean_a = {k: np.mean([r[k] for r in acc_list]) for k in keys}
    std_a = {k: np.std([r[k] for r in acc_list]) for k in keys}
    return mean_a, std_a

rows = []
for method_name in METHODS:
    acc_list = [results[method_name][s] for s in SEEDS]
    mean_a, std_a = _mean_std(acc_list)
    avg_equal = np.mean(list(mean_a.values()))
    avg_equal_std = np.std([np.mean(list(r.values())) for r in acc_list])
    avg_weighted = sample_weighted_accuracy(mean_a, client_test_sizes)
    avg_weighted_std = np.std([sample_weighted_accuracy(r, client_test_sizes) for r in acc_list])
    row = {cname: f"{100*mean_a[cname]:.1f}% ± {100*std_a[cname]:.1f}%" for cname in client_names}
    row["Avg (equal)"] = f"{100*avg_equal:.1f}% ± {100*avg_equal_std:.1f}%"
    row["Avg (weighted)"] = f"{100*avg_weighted:.1f}% ± {100*avg_weighted_std:.1f}%"
    row["_method"] = method_name
    rows.append(row)

acc_table = pd.DataFrame(rows).set_index("_method")
acc_table.index.name = "Model"
print("Test accuracy (mean ± std over seeds) for each model:\n")
display(acc_table)

Test accuracy (mean ± std over seeds) for each model:



,client_1,client_2,client_3,client_4,Avg (equal),Avg (weighted)
Model,,,,,,
Vanilla FedAvg,0.0% ± 0.0%,0.0% ± 0.0%,8.5% ± 0.0%,56.9% ± 0.0%,16.4% ± 0.0%,32.5% ± 0.0%
Local-only,0.0% ± 0.0%,100.0% ± 0.0%,49.5% ± 10.5%,14.4% ± 20.3%,41.0% ± 6.8%,32.8% ± 13.3%
Per-FedAvg,0.0% ± 0.0%,100.0% ± 0.0%,24.6% ± 22.8%,52.3% ± 6.5%,44.2% ± 4.1%,40.2% ± 7.4%
FedPer,0.0% ± 0.0%,66.7% ± 47.1%,33.3% ± 19.8%,38.0% ± 26.8%,34.5% ± 15.2%,36.2% ± 16.1%


## Ablations: effect of inner personalization steps (Per-FedAvg)

Vary number of inner-loop steps (1, 3, 5, 10) to see how personalization strength affects accuracy. Single seed for speed.

In [ ]:
INNER_STEPS_ABLATION = [1, 3, 5, 10]
ablation_results = {}  # inner_steps -> per-client accs (single seed 42)
for steps in INNER_STEPS_ABLATION:
    accs = train_per_fedavg(seed=42, inner_steps=steps)
    ablation_results[steps] = accs
    avg = np.mean(list(accs.values()))
    print(f"Inner steps = {steps}: avg acc = {100*avg:.1f}% | " + ", ".join(f"{c}: {100*accs[c]:.1f}%" for c in client_names))

Inner steps = 1: avg acc = 39.2% | client_1: 0.0%, client_2: 100.0%, client_3: 56.9%, client_4: 0.0%
Inner steps = 3: avg acc = 50.0% | client_1: 0.0%, client_2: 100.0%, client_3: 56.9%, client_4: 43.1%
Inner steps = 5: avg acc = 53.5% | client_1: 0.0%, client_2: 100.0%, client_3: 56.9%, client_4: 56.9%
Inner steps = 10: avg acc = 53.5% | client_1: 0.0%, client_2: 100.0%, client_3: 56.9%, client_4: 56.9%


## Per-class metrics (Per-FedAvg, seed 42, client_4)

Precision, recall, F1 per transport mode and confusion matrix for the largest client.

In [ ]:
# Re-run Per-FedAvg once to get model, then detailed metrics on client_4
set_seed(42)
per_fed_model = SimpleNet(input_dim, global_num_classes).to(device)
meta_opt = torch.optim.Adam(per_fed_model.parameters(), lr=META_LR)
loss_fn = nn.CrossEntropyLoss()
for epoch in range(1, META_EPOCHS + 1):
    per_fed_model.train()
    meta_opt.zero_grad()
    for cname in client_names:
        train_ds = clients[cname]["train"]
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        local = copy.deepcopy(per_fed_model).to(device)
        inner_opt = torch.optim.SGD(local.parameters(), lr=INNER_LR)
        for _ in range(INNER_STEPS):
            for xb, yb in train_loader:
                xb, yb = xb.to(device), yb.to(device)
                inner_opt.zero_grad()
                loss = loss_fn(local(xb), yb)
                loss.backward()
                inner_opt.step()
        local.train()
        inner_grads = [torch.zeros_like(p) for p in per_fed_model.parameters()]
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            local.zero_grad()
            loss = loss_fn(local(xb), yb)
            loss.backward()
            for k, p in enumerate(local.parameters()):
                if p.grad is not None:
                    inner_grads[k] = inner_grads[k] + p.grad.detach()
        n_b = len(train_loader)
        if n_b > 0:
            inner_grads = [g / n_b for g in inner_grads]
        for gp, lg in zip(per_fed_model.parameters(), inner_grads):
            if gp.grad is None:
                gp.grad = lg.clone()
            else:
                gp.grad += lg
    for p in per_fed_model.parameters():
        if p.grad is not None:
            p.grad /= len(client_names)
    meta_opt.step()

cname = "client_4"
metrics = get_detailed_metrics(
    per_fed_model, clients[cname]["test"], device,
    train_ds=clients[cname]["train"], inner_steps=INNER_STEPS, inner_lr=INNER_LR,
    num_classes=global_num_classes,
)
print(f"Per-class metrics for {cname} (mode = transport class):")
n_classes = min(global_num_classes, len(metrics["precision_per_class"]))
for idx in range(n_classes):
    mode = IDX_TO_MODE.get(idx, str(idx))
    print(f"  {mode}: P={metrics['precision_per_class'][idx]:.3f}, R={metrics['recall_per_class'][idx]:.3f}, F1={metrics['f1_per_class'][idx]:.3f}")
print("Confusion matrix (rows=true, cols=pred):")
print(metrics["confusion_matrix"])
print("(Rows/cols: car, walk, bus, train)")


Per-class metrics for client_4 (mode = transport class):
  car: P=0.000, R=0.000, F1=0.000
  walk: P=0.431, R=1.000, F1=0.602
  bus: P=0.000, R=0.000, F1=0.000
  train: P=0.000, R=0.000, F1=0.000
Confusion matrix (rows=true, cols=pred):
[[  0   0   0   0]
 [  0  87   0   0]
 [  0 115   0   0]
 [  0   0   0   0]]
(Rows/cols: car, walk, bus, train)


## Summary

- **Heterogeneity**: Label distributions and data sizes per client are reported above; small clients (e.g. client_1) have very few windows.
- **Methods**: Vanilla FedAvg, Local-only, Per-FedAvg, and FedPer are run with multiple seeds; see the **Final comparison report** for mean ± std per client and average (equal and sample-weighted) accuracies.
- **Ablations**: Effect of inner personalization steps (Per-FedAvg) is shown above.
- **Per-class metrics**: Precision, recall, F1 and confusion matrix for Per-FedAvg on client_4 are in the previous section.


---

## Unified experiments (TMDNet) — team protocol in this notebook

Same implementation as **`train_unified_fedavg_tmdnet.py`** and **`train_unified_per_fedavg_tmdnet.py`** (imported below). Outputs:

- **`results_unified/last_fedavg_tmdnet.json`** and **`last_per_fedavg_tmdnet.json`**
- **`results_unified/summary_runs.csv`** (append one row per run)

Run **IMPLEMENTATION_DIR** path cell above first (or re-run the next cell which sets path inline).


In [ ]:
# Inline path setup (safe if you skipped the earlier stub)
import sys
from pathlib import Path
_cwd = Path.cwd().resolve()
_impl = _cwd / "Personalized_Federated_Learning_Implementation"
IMPLEMENTATION_DIR = _impl if (_impl / "unified_team_protocol.py").is_file() else _cwd
assert (IMPLEMENTATION_DIR / "unified_team_protocol.py").is_file()
sys.path.insert(0, str(IMPLEMENTATION_DIR))

from unified_team_protocol import (
    TMDNet,
    UNIFIED_BATCH_SIZE,
    UNIFIED_MAX_PER_CLIENT,
    UNIFIED_LOCAL_EPOCHS,
    UNIFIED_NUM_ROUNDS,
    UNIFIED_INNER_STEPS,
    UNIFIED_INNER_LR,
    UNIFIED_META_LR,
)
print("UNIFIED_BATCH_SIZE", UNIFIED_BATCH_SIZE, "UNIFIED_NUM_ROUNDS", UNIFIED_NUM_ROUNDS)


In [ ]:
# FedAvg + TMDNet (unified)
from train_unified_fedavg_tmdnet import run_unified_fedavg

unified_fedavg_out = run_unified_fedavg(lr=0.01, seed=42, verbose=True)
unified_fedavg_out["final"]


In [ ]:
### Unified personalized model = Per-FedAvg + TMDNet

After meta-training, each client gets a **personalized** model by fine-tuning on local train data (`UNIFIED_INNER_STEPS` / `UNIFIED_INNER_LR`), then we report **test accuracy** per client and two averages:

- **`equal_weight_avg_pct`**: mean of client accuracies (each client counts equally)
- **`sample_weighted_pct`**: weighted by each client’s test set size


In [ ]:
# Per-FedAvg + TMDNet (unified personalized learning)
from train_unified_per_fedavg_tmdnet import run_unified_per_fedavg

unified_pfl_out = run_unified_per_fedavg(seed=42, verbose=True)
fin = unified_pfl_out["final"]
print("\n=== Unified personalized (Per-FedAvg + TMDNet) — test accuracy ===")
print(f"  Equal-weight average:   {fin['equal_weight_avg_pct']:.2f}%")
print(f"  Sample-weighted average: {fin['sample_weighted_pct']:.2f}%")
print("  Per client (%):", fin.get("per_client_acc_pct", {}))
fin


In [ ]:
# Load saved JSON + quick comparison table
import json
from pathlib import Path

res = IMPLEMENTATION_DIR / "results_unified"
for name in ("last_fedavg_tmdnet.json", "last_per_fedavg_tmdnet.json"):
    p = res / name
    if p.is_file():
        d = json.loads(p.read_text())
        f = d.get("final", {})
        print(name, "-> eq:", f.get("equal_weight_avg_pct"), "wtd:", f.get("sample_weighted_pct"))
    else:
        print(name, "not found yet — run cells above")

try:
    import pandas as pd
    rows = []
    for name in ("last_fedavg_tmdnet.json", "last_per_fedavg_tmdnet.json"):
        p = res / name
        if p.is_file():
            d = json.loads(p.read_text())
            rows.append({"method": d.get("method"), **d.get("final", {})})
    display(pd.DataFrame(rows)) if rows else None
except Exception as e:
    print(e)


last_fedavg_tmdnet.json not found yet — run cells above
last_per_fedavg_tmdnet.json -> eq: 28.3653 wtd: 37.8412


,method,equal_weight_avg_pct,sample_weighted_pct,per_client_acc_pct
0,per_fedavg_tmdnet,28.3653,37.8412,"{'client_1': 0.0, 'client_2': 0.0, 'client_3':..."
